In [0]:
dbutils.fs.ls("/mnt/sep2t6/default/tmp")

[FileInfo(path='dbfs:/mnt/sep2t6/default/tmp/__tmp_path_dir/', name='__tmp_path_dir/', size=0, modificationTime=1765977817000),
 FileInfo(path='dbfs:/mnt/sep2t6/default/tmp/commits/', name='commits/', size=0, modificationTime=1765977818000),
 FileInfo(path='dbfs:/mnt/sep2t6/default/tmp/metadata', name='metadata', size=45, modificationTime=1765977817000),
 FileInfo(path='dbfs:/mnt/sep2t6/default/tmp/offsets/', name='offsets/', size=0, modificationTime=1765977818000),
 FileInfo(path='dbfs:/mnt/sep2t6/default/tmp/sources/', name='sources/', size=0, modificationTime=1765977818000)]

In [0]:
from pyspark.sql.functions import col, regexp_extract, when, to_timestamp
from pyspark.sql.types import IntegerType

# 1. Checkpoint + table name
CHECKPOINT = "/mnt/sep2t6/default/tmp"
TABLE_NAME = "default.livekafka12"

# 2. Kafka options for Confluent Cloud
kafka_opts = {
    "kafka.bootstrap.servers": "pkc-s-east-2.aws.confluent.cloud:9092",  # Ensure this is correct
    "subscribe": "nov27topic",  # Ensure topic exists and is correct
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": (
        'org.apache.kafka.common.security.plain.PlainLoginModule required '
        'username=" " '
        'password=" +yCizLz5xVvYXQMPNDD97A";'
    ),
    "startingOffsets": "latest"
}


# 3. Read from Kafka (same style as your example)
raw = (
    spark.readStream
        .format("kafka")
        .options(**kafka_opts)
        .load()
)

raw.printSchema()

# 4. Convert Kafka value to string
lines = raw.selectExpr("CAST(value AS STRING) AS value", "timestamp")

# Example log:
# 205.153.190.220 - - [10/Dec/2025:20:37:56 -0500] "GET /search/tag/list HTTP/1.0" 200 5074 "http://..." "Mozilla/5.0 ..."
log_pattern = r'^(\S+) (\S+) (\S+) \[([^\]]+)\] "(\S+) (\S+)\s*(\S*)" (\d{3}) (\S+) "([^"]*)" "([^"]*)"'

parsed = lines.select(
    regexp_extract("value", log_pattern, 1).alias("ip"),
    regexp_extract("value", log_pattern, 2).alias("client_identd"),
    regexp_extract("value", log_pattern, 3).alias("user_id"),
    regexp_extract("value", log_pattern, 4).alias("datetime_str"),
    regexp_extract("value", log_pattern, 5).alias("method"),
    regexp_extract("value", log_pattern, 6).alias("endpoint"),
    regexp_extract("value", log_pattern, 7).alias("protocol"),
    regexp_extract("value", log_pattern, 8).alias("status_str"),
    regexp_extract("value", log_pattern, 9).alias("content_size_str"),
    regexp_extract("value", log_pattern, 10).alias("referrer"),
    regexp_extract("value", log_pattern, 11).alias("user_agent"),
    col("timestamp")  # Kafka message timestamp
)

# 5. Cast / clean up types
structured = (
    parsed
        .withColumn(
            "event_time",
            to_timestamp(col("datetime_str"), "dd/MMM/yyyy:HH:mm:ss Z")
        )
        .withColumn("status", col("status_str").cast(IntegerType()))
        .withColumn(
            "content_size",
            when(col("content_size_str") == "-", None)
            .otherwise(col("content_size_str").cast(IntegerType()))
        )
        .drop("status_str", "content_size_str")
)

# 6. Write stream to Delta table using availableNow + toTable (same pattern as your example)
def write_to_delta(batch_df, batch_id):
    (
        batch_df
            .write
            .format("delta")
            .mode("append")
            .saveAsTable(TABLE_NAME)
    )

# ---------------------------------------------------------
# 6. Streaming with Trigger Every 10 Seconds
# ---------------------------------------------------------
query = (
    structured.writeStream
        .outputMode("append")
        .foreachBatch(write_to_delta)
        .option("checkpointLocation", CHECKPOINT)
        .trigger(processingTime="10 seconds")   # <-- YOU REQUESTED THIS
        .start()
)

query.awaitTermination()


In [0]:
%sql
select * from livekafka12;

In [0]:
%sql
select * from livekafka12;